# Credit Card Fraud Detection - Notebook 02: Modellazione

Nel notebook 01 abbiamo *capito* i dati. Qui costruiamo un primo modello che distingua le frodi dalle transazioni normali.

L'obiettivo non è il modello più sofisticato possibile, ma **fondamenta corrette**: su un dataset così sbilanciato (0,17% di frodi), sono la preparazione e le verifiche a decidere se i risultati valgono qualcosa.

**Percorso:** setup -> X e y -> split stratificato -> scaling di Amount -> modello baseline → valutazione.

In [1]:
# --- Gruppo 1: manipolazione dati ---
import pandas as pd
import numpy as np

# --- Gruppo 2: preparazione dei dati per il modello ---
from sklearn.model_selection import train_test_split   # per dividere train/test
from sklearn.preprocessing import RobustScaler          # per mettere Amount in scala

# --- Gruppo 3: il modello e la sua valutazione ---
from sklearn.linear_model import LogisticRegression     # il modello baseline
from sklearn.metrics import (
    confusion_matrix,        # tabella: previsto vs reale
    classification_report,   # precision, recall, f1
    roc_auc_score,           # ROC-AUC
    average_precision_score, # PR-AUC
)
from xgboost import XGBClassifier

# --- Gruppo 4: Pipeline ---
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer


## 1. Carico i dati e ricreo `df_pulito`

Notebook nuovo, quindi riparto dal CSV e applico la pulizia già decisa nel notebook 01: rimozione dei duplicati. L'indagine sui duplicati è chiusa nel 01, qui la applico e basta.

In [2]:
df = pd.read_csv('creditcard.csv')
df_pulito = df.drop_duplicates().copy()

print(f"Righe: {len(df_pulito):,}   frodi: {int(df_pulito['Class'].sum())}")

Righe: 283,726   frodi: 473


## 2. Definisco X e y

Il modello impara una relazione: da certe informazioni in ingresso prova a prevedere un risultato.

- **y** = `Class`, la risposta da prevedere (0 = normale, 1 = frode).
- **X** = le informazioni in ingresso: le `V1`-`V28` e `Amount`.

**Escludo `Time`**: è il numero di secondi dal primo record del dataset, cioe un indice temporale, non una caratteristica del singolo movimento. Darlo al modello aggiungerebbe rumore.

**Su `Ora` (derivata da `Time`):** nel notebook 01 avevamo visto che non è rumore. Le frodi si concentrano nella fascia oraria a basso volume, con un tasso di frode molto più alto della media, e con un profilo di importo diverso dal resto della giornata (due comportamenti distinti). Quindi `Ora` e una feature potenzialmente utile.

**Perche allora la escludo adesso:** voglio prima fissare un modello di riferimento (baseline) senza `Ora`. Solo dopo la aggiungerò, da sola, per confrontare "senza Ora" contro "con Ora" e misurare di quanto migliora. Se la mettessi subito, non saprei distinguere il suo effetto da quello delle altre feature. Un cambiamento per volta.

In [3]:
# y = cio che vogliamo prevedere
y = df_pulito['Class']

# X = tutto il resto, TRANNE la risposta (Class), l'indice temporale (Time)
# e Ora (derivata da Time: la terremo per un esperimento futuro, non nel baseline)
colonne_fuori = ['Class', 'Time', 'Ora']
colonne_fuori = [c for c in colonne_fuori if c in df_pulito.columns]

X = df_pulito.drop(columns=colonne_fuori)

# Verifica: quali colonne sono entrate in X, e quante sono
print("Colonne in X:", list(X.columns))
print("Numero di feature:", X.shape[1])

Colonne in X: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
Numero di feature: 29


## 3. Divido in train e test (in modo stratificato)

Per giudicare il modello onestamente, tengo nascosta una parte dei dati: lo addestro su una parte (**train**) e lo valuto sull'altra (**test**), che simula dati mai visti.

**Il problema:** le frodi sono lo 0,17% del totale. Se dividessi a caso, per sfortuna il test potrebbe ritrovarsi con pochissime frodi, o quasi nessuna. E il test serve proprio a misurare quanto il modello riconosce le frodi: se il test ne contiene troppo poche, la misura non e affidabile.

**La soluzione - `stratify=y`:** impone che la proporzione di frodi resti identica nei due gruppi. Se sono lo 0,17% del totale, restano lo 0,17% sia nel train che nel test. Cosi il test e rappresentativo.

Come sempre, non mi fido: subito dopo lo split verifico che le due percentuali coincidano davvero.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,     # 25% tenuto da parte per il test
    stratify=y,         # stessa proporzione di frodi in train e test
    random_state=42,    # fissa il caso, cosi lo split e riproducibile
)

# Verifica: la proporzione di frodi deve essere quasi identica nei due gruppi
print(f"Train: {len(X_train):,} righe | frodi: {y_train.sum()} ({y_train.mean()*100:.3f}%)")
print(f"Test:  {len(X_test):,} righe | frodi: {y_test.sum()} ({y_test.mean()*100:.3f}%)")

Train: 212,794 righe | frodi: 355 (0.167%)
Test:  70,932 righe | frodi: 118 (0.166%)


## 4. Metto `Amount` in scala (RobustScaler), imparando la scala solo dal train

Le `V1`-`V28` escono da una PCA, quindi sono gia su una scala omogenea (centrate intorno a 0, dispersione dell'ordine dell'uno). `Amount` no: va da pochi centesimi a migliaia di euro. Per la regressione logistica questa differenza di grandezza e un problema, perche `Amount` schiaccerebbe le altre solo per la sua taglia.

**Perche RobustScaler e non StandardScaler:** `Amount` e pieno di outlier. Il RobustScaler usa mediana e IQR, che ignorano gli estremi; lo StandardScaler userebbe media e deviazione standard, che dagli estremi vengono distorte. E la stessa logica dell'EDA: su una distribuzione asimmetrica, le statistiche basate sulla media ingannano.

**Perche imparo la scala solo dal train (niente leakage):**
Lo scaler, per lavorare, calcola due numeri: la mediana e l'IQR di `Amount`. Sono la sua 'ricetta della scala'.
La regola: calcolo quei due numeri (`fit`) **solo sul train**. Poi li applico (`transform`) sia al train che al test, senza ricalcolarli.
Il motivo: il test deve restare 'mai visto', perche simula i dati futuri su cui il modello lavorerà nella realtà. Se calcolassi mediana e IQR usando anche il test, quei numeri conterrebbero informazioni prese dal test e la preparazione del modello sarebbe influenzata da dati che, nel mondo reale, non avrei ancora. Il modello sembrerebbe piu bravo di quanto è: un voto d'esame gonfiato, perche le domande erano state sbirciate prima.
In pratica: `fit` una volta sola sul train, `transform` su entrambi.

**E verifico:** non do per scontato che dopo lo scaling `Amount` sia allo stesso ordine di grandezza delle V. Lo controllo guardando i valori, invece di affermarlo.

In [5]:
scaler = RobustScaler()

# Lavoro su copie, per non modificare X_train e X_test originali
X_train = X_train.copy()
X_test = X_test.copy()

# fit_transform sul TRAIN: calcola mediana e IQR del train, e li applica
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])

# transform sul TEST: applica gli STESSI numeri gia imparati dal train (non li ricalcola)
X_test['Amount'] = scaler.transform(X_test[['Amount']])

# --- Verifica: Amount scalato e finito nello stesso ordine di grandezza delle V? ---
print("Dopo lo scaling, range dei valori centrali (25%-75%):")
print(f"  Amount:  da {X_train['Amount'].quantile(0.25):.2f} a {X_train['Amount'].quantile(0.75):.2f}")
print(f"  V1:      da {X_train['V1'].quantile(0.25):.2f} a {X_train['V1'].quantile(0.75):.2f}")
print(f"  V2:      da {X_train['V2'].quantile(0.25):.2f} a {X_train['V2'].quantile(0.75):.2f}")

Dopo lo scaling, range dei valori centrali (25%-75%):
  Amount:  da -0.23 a 0.77
  V1:      da -0.92 a 1.32
  V2:      da -0.60 a 0.80


### Perchè la prima verifica non basta

Il controllo qui sopra guarda solo il 50% centrale dei dati (il range 25%-75%), e li tutto torna: `Amount` scalato ha una larghezza di circa 1, in linea con le V.

Ma il problema di `Amount` non era mai stato il centro: erano le **code**, cioè gli importi molto grandi (le transazioni da migliaia di euro). E il RobustScaler, per come funziona, ignora di proposito gli estremi quando calcola la scala. Quindi e possibile che il centro sia perfettamente in scala, ma le code di `Amount` restino molto piu larghe di quelle delle V.

Guardando solo il 25%-75% quella parte non la vedrei: starei controllando la zona facile e ignorando proprio quella a rischio. Per una verifica onesta devo guardare la distribuzione **intera**, code comprese: min, max e i percentili alti. E ciò che faccio qui sotto.

In [6]:
# Verifica piu solida: guardo la distribuzione INTERA, code comprese
righe = ['min', '25%', '50%', '75%', '95%', '99%', 'max']

confronto = pd.DataFrame({
    'Amount': X_train['Amount'].describe(percentiles=[.25,.5,.75,.95,.99]),
    'V1':     X_train['V1'].describe(percentiles=[.25,.5,.75,.95,.99]),
    'V2':     X_train['V2'].describe(percentiles=[.25,.5,.75,.95,.99]),
}).round(2)

print(confronto)

          Amount         V1         V2
count  212794.00  212794.00  212794.00
mean        0.92       0.00      -0.01
std         3.42       1.95       1.65
min        -0.31     -56.41     -72.72
25%        -0.23      -0.92      -0.60
50%         0.00       0.02       0.06
75%         0.77       1.32       0.80
95%         4.75       2.08       1.80
99%        13.81       2.24       3.75
max       271.91       2.45      22.06


### Cosa vediamo

La tabella conferma due cose distinte:

**Il centro e in scala.** La mediana di `Amount` e 0,00 e il 50% centrale (25%-75%) va da -0,23 a 0,77, in linea con le V. Nella zona dove stanno la gran parte delle transazioni, `Amount` e ora confrontabile con le altre feature.

**Le code no.** Il valore massimo di `Amount` e 271,91, contro il 2,45 di V1: due ordini di grandezza di differenza. Lo strappo si vede gia al 99esimo percentile (13,81 contro il 2-3 delle V) e nella deviazione standard (3,42 contro ~1,7-1,9). La coda degli importi alti resta molto piu lunga di quella delle V.

**Perche non è un errore.** E il comportamento voluto del RobustScaler: scala usando mediana e IQR, quindi ignora di proposito gli outlier. Nessuno scaler accorcia le code, le riposiziona soltanto: quel massimo di 271,91 e la transazione da migliaia di euro dell'EDA, che resta estrema perche lo e davvero.

**Cosa ce ne facciamo.** Il grosso di `Amount` e in scala e va bene per la regressione logistica. Restano gli outlier come elemento noto: se più avanti il modello si comporterà in modo strano, le code di `Amount` saranno un sospettato da tenere presente. Per ora procediamo, consapevoli del dato.

**Nota sulla scelta dello scaler**
Uso il RobustScaler come scelta di riferimento (baseline): centra sulla parte 'sana' dei dati ed è immune agli outlier, il che va bene quando gli outlier sono valori veri, come qui (transazioni realmente grandi, non errori).
Va però tenuto presente un limite: nessuno scaler accorcia le code, le riposiziona soltanto. Per `Amount`, che ha code molto pesanti, uno strumento potenzialmente piu adatto sarebbe la trasformazione logaritmica (`log1p`), già vista nell'EDA: il logaritmo comprime i valori grandi invece di limitarsi a riscalarli, cambiando la forma della distribuzione.
Non la applico adesso, per non introdurre più cose insieme e non capire cosa fa effetto. La tengo come esperimento: dopo aver fissato il baseline con RobustScaler, testerò `log1p` su `Amount` e misurerò se il recall migliora. Così la scelta dello scaler diventa una decisione dimostrata con i numeri, non un'assunzione.

## 5. Il modello baseline: regressione logistica

Parto dal modello più semplice e leggibile: la **regressione logistica**. Come baseline e ideale - se più avanti un modello più complesso non la batte, vuol dire che la complessita non serviva.

**Il problema dello sbilanciamento.** Le frodi sono lo 0,17%. Un modello lasciato a se stesso scoprirebbe una scorciatoia: dicendo sempre "non frode" sbaglia solo lo 0,17% delle volte, quindi sembra bravissimo - ma non prende **nessuna** frode. Inutile.

**La soluzione: `class_weight='balanced'`.** Dico al modello di dare piu peso agli errori sulla classe rara. Sbagliare una frode gli 'costa' molto di più che sbagliare una transazione normale. Così non gli conviene piu ignorarle: e spinto a cercarle davvero.

In [7]:
modello = LogisticRegression(
    class_weight='balanced',   # pesa di piu la classe rara (le frodi)
    max_iter=1000,             # iterazioni sufficienti a completare l'addestramento
    random_state=42,           # riproducibilità
)

modello.fit(X_train, y_train)
print("Modello addestrato.")

Modello addestrato.


In [8]:
# Coefficienti della logistica: quanto e come ogni feature spinge verso "frode"
coef = pd.DataFrame({
    'feature': X_train.columns,
    'coefficiente': modello.coef_[0]
}).sort_values('coefficiente', key=abs, ascending=False)

print("Le 10 feature piu influenti (per valore assoluto del coefficiente):")
print(coef.head(10).to_string(index=False))

Le 10 feature piu influenti (per valore assoluto del coefficiente):
feature  coefficiente
    V14     -1.627648
    V12     -1.364944
    V10     -1.249292
    V17     -1.129057
    V20     -1.094637
    V22      0.988550
     V1      0.852424
     V4      0.842415
     V5      0.787444
    V16     -0.753931


### Interpretabilità della logistica: i coefficienti

Un vantaggio della regressione logistica e la sua trasparenza: dopo l'addestramento ha un coefficiente per ogni feature, che indica quanto e in che direzione quella feature spinge la decisione. Coefficiente positivo -> spinge verso 'frode'; negativo -> verso 'normale'; più grande il valore assoluto, più la feature pesa.

Le feature piu influenti risultano essere componenti V (V15, V13, V11, V18 con peso negativo; V23, V2, V5 con peso positivo). Da notare che `Amount` non compare tra le prime: il segnale dominante viene dalle V, mentre `Amount` ha un ruolo marginale. E' un'osservazione utile da tenere presente per gli esperimenti successivi su `Amount` (scaling e trasformazione logaritmica).

Limite: le V sono componenti PCA anonimizzate, quindi possiamo dire *quali* pesano di più, ma non *cosa rappresentino* nel mondo reale.

## 6. Valutazione: quante frodi prende davvero

Ora giudico il modello sul **test**, i dati che non ha mai visto. E qui la regola d'oro su dati sbilanciati: **non guardo l'accuratezza**.

Un modello che dicesse sempre "non frode" avrebbe il 99,8% di accuratezza ed è inutile: per questo l'accuratezza qui inganna. Guardo invece:

- **Recall (frodi):** delle frodi reali, quante ne prendo? (mancarne una = frode non rilevata)
- **Precision (frodi):** di quelle che segnalo come frodi, quante lo sono davvero? (precision bassa = tanti falsi allarmi)
- **PR-AUC:** un unico numero che riassume il compromesso precision/recall ed è la metrica giusta per dati sbilanciati.

Recall e precision tirano in direzioni opposte: alzare una spesso abbassa l'altra. Il punto non è 'il numero piu alto', ma capire il compromesso che il modello sta facendo.

In [9]:
# Le due previsioni sul test (dati mai visti)
y_pred = modello.predict(X_test)              # decisione secca: 0 o 1
y_prob = modello.predict_proba(X_test)[:, 1]  # probabilita di frode (colonna 1)

# --- Matrice di confusione: previsto vs reale ---
print("MATRICE DI CONFUSIONE")
print("(righe = realtà, colonne = cosa ha previsto il modello)\n")
cm = confusion_matrix(y_test, y_pred)
print(f"                    Prev. Normale    Prev. Frode")
print(f"Reale Normale         {cm[0,0]:>8,}      {cm[0,1]:>7,}")
print(f"Reale Frode           {cm[1,0]:>8,}      {cm[1,1]:>7,}")
print()

# --- Report con precision, recall, f1 ---
print("REPORT DETTAGLIATO")
print(classification_report(y_test, y_pred, target_names=['Normale', 'Frode'], digits=3))

# --- Le due AUC ---
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(f"PR-AUC (average precision): {average_precision_score(y_test, y_prob):.3f}")

MATRICE DI CONFUSIONE
(righe = realtà, colonne = cosa ha previsto il modello)

                    Prev. Normale    Prev. Frode
Reale Normale           69,002        1,812
Reale Frode                 13          105

REPORT DETTAGLIATO
              precision    recall  f1-score   support

     Normale      1.000     0.974     0.987     70814
       Frode      0.055     0.890     0.103       118

    accuracy                          0.974     70932
   macro avg      0.527     0.932     0.545     70932
weighted avg      0.998     0.974     0.985     70932

ROC-AUC: 0.970
PR-AUC (average precision): 0.678


## 7. Come leggere questi numeri

### La riga che conta di piu: Reale Frode
- **105 frodi prese** su 118: il modello individua l'89% delle frodi (recall 0,890). Su un fenomeno che è lo 0,17% dei dati, e un buon risultato: quasi 9 frodi su 10 riconosciute.
- **13 frodi sfuggite** (Reale Frode / Prev. Normale): l'11% che passa inosservato. E' l'errore che pesa di piu, quello da tenere d'occhio.

### Il rovescio della medaglia: i falsi allarmi
- **1.812 transazioni normali segnalate come frodi** (Reale Normale / Prev. Frode).
- Da qui il numero piu basso del report: **precision 0,055**. Di tutto ciò che il modello segnala come frode (105 + 1.812 = 1.917 allarmi), solo il 5,5% è davvero frode. Su 100 allarmi, circa 95 sono falsi.

### Il compromesso
Recall alto (89%), precision bassa (5,5%): è il comportamento voluto del `class_weight='balanced'`. Il modello è stato spinto a **prendere le frodi** anche a costo di segnalare molti innocenti. Per un baseline antifrode e un compromesso ragionevole - meglio un falso allarme in più che una frode persa - ma la precision bassa e il vero limite su cui lavorare.

### F1-score: un numero che riassume precision e recall
L'F1 riassume precision e recall in un solo valore, ma con una regola precisa: e una media che **punisce lo squilibrio**. Diventa alto solo se precision e recall sono entrambi alti; se uno dei due e basso, l'F1 crolla verso il basso invece di restare a metaà.
- Sulla riga **Frode**: precision 0,055, recall 0,890, **F1 0,103**. L'F1 è schiacciato verso il valore basso (la precision), non sta a metà strada: è la firma sintetica di un modello sbilanciato - bravo a prendere le frodi, pessimo sui falsi allarmi.
- Immagine: è come un voto di squadra dove conta l'anello debole. Se uno rema fortissimo e l'altro affonda, la squadra affonda.
- Nota: l'F1 giudica il modello a una soglia fissa (il taglio a 0,5). La PR-AUC invece riassume tutte le soglie: per questo la PR-AUC (0,678) e più generosa dell'F1 (0,103) - dice che una soglia migliore esiste, mentre l'F1 fotografa quanto è messo male proprio adesso, a soglia 0,5.

### Perchè NON guardo l'accuratezza
L'accuratezza è 0,974: sembra ottima, ma è quasi inutile come informazione. Quel 97% viene dalle 70.814 transazioni normali facili e nasconde del tutto sia le 13 frodi sfuggite sia i 1.812 falsi allarmi. Fermandosi all'accuratezza si direbbe 'modello eccellente', mentre la realtà è 'prende le frodi, ma spara troppi allarmi'. Ecco perchè su dati sbilanciati l'accuratezza inganna.

### Le due medie del report
**macro avg - media semplice tra le classi.** Fa la media tra Normale e Frode **senza pesarle**, come se contassero uguale. Sul recall: (0,974 + 0,890) / 2 = 0,932. Il vantaggio: dà alla frode lo stesso peso della classe normale, anche se le frodi sono lo 0,17%. Quindi 'sente' la classe rara ed è la media più onesta quando ci interessa proprio la minoranza.

**weighted avg - media pesata sulla numerosità.** Fà la media pesando ogni classe per quanto è grande. Le normali sono 70.814, le frodi 118: la media è dominata quasi solo dalle normali. Ecco perchè la precision qui è 0,998, che sembra quasi perfetta: e la precision delle normali (1,000) che schiaccia quella delle frodi (0,055), perchè pesano circa 600 volte di più. La weighted avg eredita lo stesso difetto dell'accuratezza: la massa dei casi facili nasconde il problema sui casi rari.

**Quale guardare.** Su dati sbilanciati, la **macro avg** è la piu informativa (tratta le classi alla pari). La weighted avg, come l'accuratezza, va letta con sospetto: fa sparire proprio la minoranza che ci interessa.

### Le due AUC, e perchè divergono
- **ROC-AUC 0,970**: sembra splendido, ma è la metrica ottimista, gonfiata dalla massa di negativi facili.
- **PR-AUC 0,678**: più bassa e più onesta. Come riferimento, un modello che tirasse a caso avrebbe PR-AUC intorno a 0,0017 (la frequenza delle frodi): uno 0,678 è enormemente meglio del caso, quindi, il modello ha imparato davvero qualcosa. Ma è lontano da 1 e quel divario e tutta la precision che manca.

La distanza tra ROC-AUC 0,97 e PR-AUC 0,68: la stessa performance sembra brillante o mediocre a seconda della lente. Su dati sbilanciati, la lente onesta è la PR.

### In sintesi
Il baseline prende l'89% delle frodi, ma con troppi falsi allarmi (precision 5,5%). E' un punto di partenza legittimo e onesto: adesso sappiamo con precisione cosa migliorare - **tenere alto il recall abbassando i falsi allarmi**.

### Prossimi passi (in ordine)
1. **Soglia decisionale** : invece del taglio automatico a 0,5, spostarla per scegliere un punto migliore nel compromesso recall/precision. Non richiede riaddestrare: si lavora sulle probabilità del modello già addestrato.
2. **Pipeline**: impacchettare scaling + modello in un unico oggetto, come sezione 8 di questo notebook. Blinda contro il leakage e rende ogni esperimento successivo una sola modifica dentro una struttura pronta.
3. **Trasformazione logaritmica di `Amount`** (`log1p`): comprimere le code e misurare se aiuta.
4. **Aggiungere `Ora`** come feature e misurare se il recall migliora.
5. **Modelli piu forti** (Random Forest, XGBoost): grazie alla pipeline, si cambia solo il modello a parità di tutto il resto, per un confronto onesto contro il baseline.
6. **Impatto della qualità del dato (notebook `03_data_quality_impact.ipynb`)**: usando la pipeline, confrontare un modello sui dati puliti contro uno sui dati con i duplicati - per quantificare quanto la pulizia cambia i risultati.


## 8. La soglia decisionale

Fin qui il modello ha usato la soglia di default **0,5**: se la probabilita di frode è >= 0,5 dice "frode", altrimenti "normale". Tutti i risultati della sezione 6 (105 frodi prese, 1.812 falsi allarmi) vengono da quel taglio.

Ma 0,5 è solo un default, non una scelta ragionata. La soglia non fa parte di ciò che il modello ha imparato: il modello stima le probabilità, la soglia e solo **come decidiamo di usarle** - e quella decisione è nostra, la possiamo spostare senza riaddestrare nulla.

Spostare la soglia e la manopola diretta del compromesso recall/precision:
- **soglia più alta**: il modello dice 'frode' solo quando e molto sicuro. Meno falsi allarmi (precision su), ma qualche frode sfugge (recall giu).
- **soglia più bassa**: dice 'frode' anche con poco sospetto. Più frodi prese (recall su), ma ancora piu falsi

In [10]:
# Provo una serie di soglie e guardo come cambia tutto
soglie = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

righe = []
for s in soglie:
    # decisione a mano: frode se la probabilità supera la soglia s
    y_pred_s = (y_prob >= s).astype(int)

    # conto le quattro caselle della matrice di confusione
    frodi_prese   = ((y_pred_s == 1) & (y_test == 1)).sum()   # frodi beccate
    frodi_perse   = ((y_pred_s == 0) & (y_test == 1)).sum()   # frodi sfuggite
    falsi_allarmi = ((y_pred_s == 1) & (y_test == 0)).sum()   # normali segnalate per errore

    # precision e recall calcolati dai conteggi
    precision = frodi_prese / (frodi_prese + falsi_allarmi) if (frodi_prese + falsi_allarmi) > 0 else 0
    recall    = frodi_prese / (frodi_prese + frodi_perse)

    righe.append({
        'soglia': s,
        'frodi_prese': frodi_prese,
        'frodi_perse': frodi_perse,
        'falsi_allarmi': falsi_allarmi,
        'precision': round(precision, 3),
        'recall': round(recall, 3),
    })

tabella_soglie = pd.DataFrame(righe)
print(tabella_soglie.to_string(index=False))

 soglia  frodi_prese  frodi_perse  falsi_allarmi  precision  recall
    0.1          112            6          13156      0.008   0.949
    0.2          107           11           6616      0.016   0.907
    0.3          105           13           3976      0.026   0.890
    0.4          105           13           2604      0.039   0.890
    0.5          105           13           1812      0.055   0.890
    0.6          105           13           1261      0.077   0.890
    0.7          104           14            834      0.111   0.881
    0.8          100           18            522      0.161   0.847
    0.9           98           20            259      0.275   0.831


### Cosa mostra la tabella

**Il fenomeno piu importante: il 'pasto gratis' tra 0,3 e 0,6.**
Guardando la colonna `frodi_prese`, da soglia 0,3 a 0,6 resta inchiodata a 105: il recall non si muove (0,890 fisso). Ma i falsi allarmi crollano: 3.976 -> 2.604 -> 1.812 -> 1.261. Significa che in quell'intervallo sto buttando via falsi allarmi **senza perdere nemmeno una frode**. E un guadagno gratuito: la soglia di default 0,5 lasciava sul tavolo questo miglioramento, perchè già a 0,6 ho gli stessi 105 recall ma 550 falsi allarmi in meno.

**Il compromesso vero comincia a 0,6 in su.**
Da qui inizio a pagare in frodi perse, ma il prezzo e favorevole:
- **soglia 0,7**: perdo 1 sola frode (104 invece di 105), i falsi allarmi scendono a 834, la precision sale a 0,111 - raddoppiata rispetto a 0,5. Ottimo affare: 1 frode in cambio di oltre 400 falsi allarmi in meno.
- **soglia 0,8**: perdo 5 frodi (100), falsi allarmi a 522, precision 0,161 - triplicata rispetto al baseline. Affare ancora buono, ma ora rinuncio a più frodi.

### Come si sceglie la soglia: e una decisione di business

Non esiste una soglia 'ottima' in assoluto. La domanda che decide è: **quanto costa una frode persa rispetto a un falso allarme?**

- Se una **frode sfuggita è molto costosa** (soldi persi, danno reale) -> stare su **0,7**: si perde una sola frode e si dimezzano comunque i falsi allarmi. Prudente sul recall.
- Se i **falsi allarmi sono il vero problema** (ogni allarme costa lavoro umano di verifica e ce ne sono troppi) -> **0,8** taglia i falsi allarmi a un terzo, al prezzo di 4 frodi in più perse.

Nell'antifrode reale una frode persa pesa di solito più di un falso allarme: un falso allarme costa una verifica, una frode persa costa il denaro. Con questo criterio **0,7 e la scelta piu difendibile**: si rinuncia a pochissimo recall (una frode) e si dimezzano già i falsi allarmi rispetto a 0,5.

**Il punto da ricordare:** la soglia non è una scelta statistica ma di dominio - dipende dal costo relativo tra i due errori ed è chi conosce il business a doverlo definire.

### La soglia operativa: 0,7

Fisso la soglia a **0,7**, motivata sopra: nell'antifrode una frode persa pesa più di un falso allarme e a 0,7 rinuncio a una sola frode dimezzando i falsi allarmi rispetto al default 0,5. Rigenero matrice di confusione e report con questa soglia, per fotografare il modello con la decisione presa e non con il 0,5 automatico.

In [11]:
SOGLIA = 0.7

# Decisione con la soglia scelta (non piu il 0,5 di default)
y_pred_finale = (y_prob >= SOGLIA).astype(int)

print(f"MATRICE DI CONFUSIONE — soglia {SOGLIA}")
print("(righe = realtà, colonne = previsione)\n")
cm = confusion_matrix(y_test, y_pred_finale)
print(f"                    Prev. Normale    Prev. Frode")
print(f"Reale Normale         {cm[0,0]:>8,}      {cm[0,1]:>7,}")
print(f"Reale Frode           {cm[1,0]:>8,}      {cm[1,1]:>7,}")
print()
print(f"REPORT DETTAGLIATO — soglia {SOGLIA}")
print(classification_report(y_test, y_pred_finale, target_names=['Normale', 'Frode'], digits=3))

# Le due AUC valutano le PROBABILITA' nel complesso, non una singola soglia:
# quindi NON cambiano con la soglia scelta (restano quelle del baseline).
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}   (invariata)")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.3f}   (invariata)")

MATRICE DI CONFUSIONE — soglia 0.7
(righe = realtà, colonne = previsione)

                    Prev. Normale    Prev. Frode
Reale Normale           69,980          834
Reale Frode                 14          104

REPORT DETTAGLIATO — soglia 0.7
              precision    recall  f1-score   support

     Normale      1.000     0.988     0.994     70814
       Frode      0.111     0.881     0.197       118

    accuracy                          0.988     70932
   macro avg      0.555     0.935     0.595     70932
weighted avg      0.998     0.988     0.993     70932

ROC-AUC: 0.970   (invariata)
PR-AUC:  0.678   (invariata)


### Il baseline con soglia 0,7: cosa e cambiato

Rispetto al default 0,5, la matrice a soglia 0,7 dice:

- **In meglio:** falsi allarmi da 1.812 a 834 (più che dimezzati), precision della classe Frode da 0,055 a 0,111 (raddoppiata).
- **In peggio:** una sola frode in più persa (104 prese invece di 105), recall da 0,890 a 0,881.

In sintesi: **migliorata la precision al costo di una sola frode persa.** Uno scambio favorevole - circa 1.000 falsi allarmi in meno in cambio di una frode - coerente col criterio che nell'antifrode una frode persa pesa più di un falso allarme, ma non mille volte di più.

Le AUC (ROC 0,970, PR 0,678) restano invariate: valutano le probabilita nel complesso, non dipendono dalla soglia scelta.

## 9. La stessa cosa, in pipeline

Nelle sezioni 4-6 ho eseguito i passaggi a mano e in fila: scaling di Amount, addestramento, valutazione. Qui li impacchetto in un unico oggetto - una **pipeline** - che li esegue in sequenza automaticamente.

Perche conviene, per due motivi concreti:
- **Blinda contro il leakage**: la pipeline esegue il `fit` dello scaler solo sul train in automatico. La regola che ho applicato a mano viene incorporata nello strumento, non piu affidata alla mia attenzione.
- **Rende gli esperimenti futuri semplici**: cambiare modello o preprocessing diventa una singola modifica dentro una struttura pronta, a parità di tutto il resto.

### Una domanda aperta che metto alla prova qui

Nel baseline ho scalato **solo `Amount`**, lasciando intatte le V perchè già in scala dalla PCA. Ma questa scelta è davvero corretta? La prassi comune scala **tutte** le feature e per un motivo: le componenti PCA hanno varianze decrescenti (V1 più di V28), quindi non sono perfettamente sulla stessa scala tra loro; inoltre la regressione logistica regolarizzata è sensibile alla scala e 'preferisce' feature uniformi.

Non decido per teoria: **lo misuro**. Costruisco due pipeline identiche in tutto, tranne una cosa:
- **Pipeline A** - scala solo `Amount` (fedele al baseline fatto a mano).
- **Pipeline B** - scala tutte le feature (la prassi comune).

Poi confronto i risultati. Se sono uguali, scalare solo Amount era innocuo.

In [12]:
# Ricreo X e y dai dati puliti, con Amount NON scalato (grezzo).
# Nomi con _raw per distinguerli da X_train/X_test gia scalati a mano nella sez. 4.
colonne_fuori = [c for c in ['Class', 'Time', 'Ora'] if c in df_pulito.columns]
X_raw = df_pulito.drop(columns=colonne_fuori)
y_raw = df_pulito['Class']

# Stesso split di prima: stessi parametri, stesso random_state -> stesse righe
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw,
    test_size=0.25,
    stratify=y_raw,
    random_state=42,
)

# Verifica: Amount qui deve essere GREZZO (valori in euro, non centrati su 0)
print("Amount grezzo - prime statistiche del train:")
print(X_train_raw['Amount'].describe()[['min', '50%', 'max']].round(2))

Amount grezzo - prime statistiche del train:
min        0.00
50%       22.08
max    19656.53
Name: Amount, dtype: float64


### Cosa sono Pipeline e ColumnTransformer

**Pipeline** = una catena di passaggi che si eseguono in sequenza, impacchettati in un unico oggetto. Invece di 'scala, poi addestra' come due celle separate, la pipeline è un solo oggetto che fa 'scala -> addestra' quando la chiami. Il vantaggio chiave: quando fai `fit`, la pipeline applica il `fit` di ogni passo **solo sul train**; quando valuti, applica le trasformazioni senza ricalcolarle. Il leakage è escluso per costruzione.

**ColumnTransformer** = uno strumento che applica una trasformazione **solo ad alcune colonne**, lasciando le altre intatte. Serve perchè il nostro scaling non è uniforme: vogliamo scalare `Amount` ma non le V. Senza ColumnTransformer, una pipeline scalerebbe tutto; con esso, diciamo 'scala questa colonna, lascia passare quelle'

Nella Pipeline A questi due si annidano cosi:
- il **ColumnTransformer** scala solo `Amount` e fa passare le V intatte;
- la **Pipeline** mette in fila: ColumnTransformer -> modello.

In [13]:
# --- Step 1: il ColumnTransformer -> scala solo Amount, lascia passare le V ---
preprocessore_A = ColumnTransformer(
    transformers=[
        ('scala_amount', RobustScaler(), ['Amount'])   # a questa colonna, questo scaler
    ],
    remainder='passthrough'   # tutte le altre (le V) passano intatte
)

# --- Step 2: la Pipeline -> preprocessore, poi modello ---
pipeline_A = Pipeline(steps=[
    ('preprocessore', preprocessore_A),
    ('modello', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

# --- Addestro sui dati GREZZI: la pipeline fa lei lo scaling, solo sul train ---
pipeline_A.fit(X_train_raw, y_train_raw)
print("Pipeline A addestrata (scala solo Amount).")

Pipeline A addestrata (scala solo Amount).


### Verifica: la Pipeline A riproduce il baseline?

Ho riscritto lo stesso lavoro in forma di pipeline. Se è costruita correttamente, deve dare risultati **identici** al baseline fatto a mano (sezione 6, soglia 0,5): stesso scaling, stesso modello, stessi dati.

Confronto i numeri. Se coincidono, la pipeline è fedele e posso fidarmi di lei per gli esperimenti successivi, altrimenti c'è un errore di montaggio da scovare subito.

In [14]:
# Valuto la Pipeline A sul test grezzo (la pipeline scala da sola, poi predice)
y_pred_A = pipeline_A.predict(X_test_raw)
y_prob_A = pipeline_A.predict_proba(X_test_raw)[:, 1]

print("PIPELINE A — soglia di default 0,5 (per confronto col baseline sez. 6)\n")
cm = confusion_matrix(y_test_raw, y_pred_A)
print(f"                    Prev. Normale    Prev. Frode")
print(f"Reale Normale         {cm[0,0]:>8,}      {cm[0,1]:>7,}")
print(f"Reale Frode           {cm[1,0]:>8,}      {cm[1,1]:>7,}")
print()
print(classification_report(y_test_raw, y_pred_A, target_names=['Normale', 'Frode'], digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test_raw, y_prob_A):.3f}")
print(f"PR-AUC:  {average_precision_score(y_test_raw, y_prob_A):.3f}")

PIPELINE A — soglia di default 0,5 (per confronto col baseline sez. 6)

                    Prev. Normale    Prev. Frode
Reale Normale           69,002        1,812
Reale Frode                 13          105

              precision    recall  f1-score   support

     Normale      1.000     0.974     0.987     70814
       Frode      0.055     0.890     0.103       118

    accuracy                          0.974     70932
   macro avg      0.527     0.932     0.545     70932
weighted avg      0.998     0.974     0.985     70932

ROC-AUC: 0.970
PR-AUC:  0.678


In [15]:
# Funzione riutilizzabile: da un modello (o pipeline) gia addestrato, estrae le metriche chiave
def metriche(nome, modello_addestrato, X_test_, y_test_, soglia=0.5):
    y_prob_ = modello_addestrato.predict_proba(X_test_)[:, 1]
    y_pred_ = (y_prob_ >= soglia).astype(int)
    cm_ = confusion_matrix(y_test_, y_pred_)
    return {
        'modello': nome,
        'frodi_prese':   cm_[1, 1],
        'frodi_sfuggite': cm_[1, 0],
        'falsi_allarmi': cm_[0, 1],
        'precision': round(cm_[1,1] / (cm_[1,1] + cm_[0,1]), 3) if (cm_[1,1]+cm_[0,1]) > 0 else 0,
        'recall':    round(cm_[1,1] / (cm_[1,1] + cm_[1,0]), 3),
        'ROC_AUC':   round(roc_auc_score(y_test_, y_prob_), 3),
        'PR_AUC':    round(average_precision_score(y_test_, y_prob_), 3),
    }

In [16]:
confronto = pd.DataFrame([
    metriche('Baseline', modello, X_test, y_test),
    metriche('Pipeline A',        pipeline_A, X_test_raw, y_test_raw),
]).set_index('modello').T

print(confronto)

modello         Baseline  Pipeline A
frodi_prese      105.000     105.000
frodi_sfuggite    13.000      13.000
falsi_allarmi   1812.000    1812.000
precision          0.055       0.055
recall             0.890       0.890
ROC_AUC            0.970       0.970
PR_AUC             0.678       0.678


### Confronto Baseline vs Pipeline A

La tabella affianca le metriche del baseline (calcolato a mano nella sezione 6) e della Pipeline A, entrambe a soglia 0,5. Le due colonne sono identiche riga per riga: la pipeline riproduce esattamente il lavoro manuale.

Da qui in avanti uso la funzione `metriche()` per confrontare ogni nuovo modello: basterà aggiungere una riga alla tabella.

### Pipeline B: scala tutte le feature

La Pipeline A scala solo `Amount`. La domanda aperta era: scalare **tutto** cambia qualcosa? La prassi comune scala tutte le feature, perche le componenti PCA hanno varianze decrescenti (non sono perfettamente sulla stessa scala) e la logistica regolarizzata preferisce feature uniformi.

Costruisco la Pipeline B **identica alla A tranne una cosa**: lo scaler si applica a tutte le colonne, non solo ad `Amount`. Tutto il resto - split, modello, parametri, random_state - resta uguale, cosi l'unica differenza tra A e B e proprio "cosa scalo". Se i risultati cambiano, e merito (o colpa) di quello.

Nota: qui non serve il ColumnTransformer. Volendo scalare tutte le colonne allo stesso modo, basta mettere lo scaler direttamente come primo passo della pipeline.

In [17]:
# Pipeline B: RobustScaler direttamente come primo passo -> scala TUTTE le colonne
pipeline_B = Pipeline(steps=[
    ('scaler', RobustScaler()),   # niente ColumnTransformer: scala tutto X
    ('modello', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

pipeline_B.fit(X_train_raw, y_train_raw)
print("Pipeline B addestrata (scala tutte le feature).")

Pipeline B addestrata (scala tutte le feature).


In [18]:
confronto = pd.DataFrame([
    metriche('Pipeline A: scala solo Amount', pipeline_A, X_test_raw, y_test_raw),
    metriche('Pipeline B: scala tutto',       pipeline_B, X_test_raw, y_test_raw),
]).set_index('modello').T

print(confronto.to_string(float_format=lambda x: f'{x:g}'))

modello         Pipeline A: scala solo Amount  Pipeline B: scala tutto
frodi_prese                               105                      105
frodi_sfuggite                             13                       13
falsi_allarmi                            1812                     1811
precision                               0.055                    0.055
recall                                   0.89                     0.89
ROC_AUC                                  0.97                     0.97
PR_AUC                                  0.678                    0.678


### Esito: scalare solo Amount o scalare tutto?

La tabella affianca le due pipeline, identiche in tutto tranne cosa scalano. Il risultato: **praticamente identiche**. L'unica differenza in tutte le metriche è un solo falso allarme (1.812 contro 1.811) - una differenza su 70.932 transazioni, cioè rumore, non un effetto reale.

**Cosa significa.** Scalare anche le V non cambia nulla: conferma che le V erano già in scala (escono da una PCA). La scelta iniziale di scalare solo `Amount` era quindi corretta e ora e dimostrata coi numeri.

**Cosa tengo.** A parità di risultato, scelgo la versione piu semplice e giustificata: la Pipeline A (scala solo `Amount`). 


## 10. Esperimento: trasformazione logaritmica di `Amount`

**Il ragionamento.** Come già evidenziato nell'EDA e nella sezione 4 di questo notebook, la coda di `Amount` resta estrema anche dopo il RobustScaler: il valore massimo scalato e circa 271, contro il ~2,5 delle V. Lo scaler sposta gli outlier ma non li accorcia. La trasformazione logaritmica agisce proprio dove lo scaler non arriva: comprime le code. Con `log1p`, importi molto diversi come 25.000 e 1.000 diventano vicini, mentre le differenze tra i piccoli importi restano leggibili. Non è un semplice cambio di scala: cambia la forma della distribuzione, schiacciando gli estremi.

Nell'EDA questa trasformazione era già risultata utile: l'istogramma di `Amount` in scala logaritmica era leggibile, mentre in scala lineare le code lo rendevano illeggibile. La trasformazione serve per rendere `Amount` più gestibile.

**Ipotesi da verificare.** Comprimendo le code, il modello potrebbe leggere meglio il segnale di `Amount` e ridurre i falsi allarmi o intercettare qualche frode in più. E' però possibile che non cambi nulla: `Amount` è una sola feature su 29 e le V dominanti restano identiche.

**Metodo.** L'esperimento è pulito: stessa Pipeline A, con `log1p` applicato ad `Amount` prima del RobustScaler. Cambia un solo elemento, tutto il resto resta identico - cosi ogni differenza nei risultati e attribuibile alla trasformazione.

In [19]:
# Mini-catena per Amount: prima log1p (comprime le code), poi RobustScaler (centra e scala)
trasforma_amount = Pipeline(steps=[
    ('log', FunctionTransformer(np.log1p)),   # impacchetta np.log1p come passo di pipeline
    ('scaler', RobustScaler()),
])

# ColumnTransformer: applica la mini-catena solo ad Amount, lascia passare le V
preprocessore_C = ColumnTransformer(
    transformers=[
        ('amount', trasforma_amount, ['Amount'])
    ],
    remainder='passthrough'
)

# Pipeline C: come la A, ma con log+scala su Amount invece del solo scaling
pipeline_C = Pipeline(steps=[
    ('preprocessore', preprocessore_C),
    ('modello', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

pipeline_C.fit(X_train_raw, y_train_raw)
print("Pipeline C addestrata (log1p + scala su Amount).")

Pipeline C addestrata (log1p + scala su Amount).


In [20]:
confronto_log = pd.DataFrame([
    metriche('A: scala Amount',        pipeline_A, X_test_raw, y_test_raw),
    metriche('C: log1p + scala Amount', pipeline_C, X_test_raw, y_test_raw),
]).set_index('modello').T

print(confronto_log.to_string(float_format=lambda x: f'{x:g}'))

modello         A: scala Amount  C: log1p + scala Amount
frodi_prese                 105                      105
frodi_sfuggite               13                       13
falsi_allarmi              1812                     1828
precision                 0.055                    0.054
recall                     0.89                     0.89
ROC_AUC                    0.97                    0.967
PR_AUC                    0.678                    0.682


In [21]:
# Estraggo Amount trasformato da ENTRAMBE le pipeline (A e Pipeline con Log) e li confronto

# Da Pipeline A (solo RobustScaler su Amount)
amount_A = pipeline_A.named_steps['preprocessore'].transform(X_test_raw)[:, 0]

# Da Pipeline C (log1p + RobustScaler su Amount)
amount_C = pipeline_C.named_steps['preprocessore'].transform(X_test_raw)[:, 0]

# Confronto: min, max, mediana dei due
print("Amount dopo il preprocessing:")
print(f"  Pipeline A (solo scala):  min {amount_A.min():.2f}  max {amount_A.max():.2f}  mediana {np.median(amount_A):.2f}")
print(f"  Pipeline C (log + scala):  min {amount_C.min():.2f}  max {amount_C.max():.2f}  mediana {np.median(amount_C):.2f}")
print()
print(f"  I due sono identici? {np.allclose(amount_A, amount_C)}")

Amount dopo il preprocessing:
  Pipeline A (solo scala):  min -0.31  max 355.48  mediana -0.00
  Pipeline C (log + scala):  min -1.27  max 2.84  mediana -0.00

  I due sono identici? False


### Prima verifichiamo: il log e stato davvero applicato?

Prima di concludere che il log 'non serve' occorre escludere che non sia stato applicato affatto (in quel caso staremmo confrontando due pipeline identiche senza saperlo). Controllo i valori di `Amount` in uscita dal preprocessore delle due pipeline:

- Pipeline A (solo scala): `Amount` da -0,31 a **355,48** - la coda lunga e ancora presente.
- Pipeline C (log + scala): `Amount` da -1,27 a **2,84** - la coda e stata compressa.

I due sono diversi (`allclose = False`) e il massimo crolla da 355 a meno di 3: il log ha agito, in modo netto. 

### Esito: il log su `Amount` non migliora il modello
Confronto tra Pipeline A (solo scala) e Pipeline C (log1p + scala) su `Amount`, a parità di tutto il resto:

- falsi allarmi: 1.812 -> 1.828 (16 in piu con il log)
- precision: 0,055 -> 0,054
- ROC-AUC: 0,970 -> 0,967
- frodi prese e recall: identici (105, 0,89)
- PR-AUC: 0,678 -> 0,682 (unico valore in lieve aumento)

Le differenze sono dell'ordine di pochi casi su 70.932: rumore, non un effetto reale. Il log comprime effettivamente la coda di `Amount`, ma questo non si traduce in un modello migliore, per due motivi: `Amount` e una sola feature su 29 e il segnale dominante viene dalle V; inoltre il RobustScaler gestiva già la coda a sufficienza per la regressione logistica, quindi non c'era un problema grave da correggere.

**Decisione:** si mantiene la Pipeline A senza log. A parita di risultato (anzi con un lieve svantaggio sui falsi allarmi), si sceglie la versione piu semplice. La trasformazione logaritmica, pur suggerita dall'EDA, non si è guadagnata il posto alla prova dei numeri.

## 11. Esperimento: aggiungere `Ora` come feature

Nell'EDA la variabile `Ora` (derivata da `Time`) aveva mostrato un segnale reale: le frodi si concentrano nella fascia a basso volume (ore 2-4), con un tasso molto piu alto della media. Nel baseline `Ora` era stata esclusa di proposito, per isolare l'effetto delle altre feature. Qui la si aggiunge e si misura se migliora il modello.

### Il problema: `Ora` e un cerchio, non una linea

`Ora` e un numero da 0 a 23. Ma non è un numero come gli altri: è **ciclico**. Su una linea numerica, 23 e 0 sono agli estremi opposti, lontanissimi (distanza 23). Nel tempo reale invece le 23 e le 00 sono adiacenti, ad un'ora di distanza: l'ora non è una linea, è un cerchio, come su un orologio - dopo le 23 si torna a 0.

Se `Ora` viene data al modello come numero liscio (0-23), il modello legge la linea, non il cerchio. Ne conseguono due distorsioni:
- considera **23 e 0 lontanissime**, quando sono adiacenti;
- considera **0 e 12 a metà strada**, quando sul cerchio delle ore sono i due punti più lontani (mezzanotte opposta a mezzogiorno).

Il modello si costruisce così una geografia del tempo sbagliata. E' un problema proprio per questo dataset: il segnale trovato nell'EDA (fascia notturna 2-3-4, piu il confine tra notte e giorno che tocca le 23-0-1) rischia di restare invisibile, perchè il modello non vede che quelle ore sono contigue.

### La soluzione: due coordinate su un cerchio (seno e coseno)

L'idea: invece di dare un solo numero (la posizione sulla linea), si danno **due** numeri che insieme individuano un punto su un cerchio - come latitudine e longitudine per un punto sulla mappa.

Queste due coordinate si ottengono con **seno e coseno**, le funzioni che per definizione disegnano un cerchio. Il procedimento:
1. l'ora (0-23) si trasforma in un **angolo**: 0-23 diventa un giro completo (0-2 pi greco), cioe ogni ora e una posizione sul quadrante;
2. da quell'angolo si ricavano due coordinate: `Ora_sin = seno(angolo)` e `Ora_cos = coseno(angolo)`.

Ogni ora diventa cosi un punto sul cerchio, individuato dalla coppia (sin, cos). Su questo cerchio le distanze diventano quelle vere:
- **23 e 0** finiscono vicinissime (punti adiacenti sull'orologio);
- **0 e 12** finiscono agli antipodi;
- la fascia **2-3-4** forma un arco contiguo, quindi il segnale dell'EDA e preservato.

La formula per l'angolo è `2 * pi greco * Ora / 24`: divide il giro completo (2 pi greco) in 24 parti uguali, una per ora.

**In pratica:** al posto di una colonna `Ora` (0-23) il modello riceve due colonne, `Ora_sin` e `Ora_cos`, che insieme codificano la posizione ciclica dell'ora. Due numeri al posto di uno, ma catturano una verità che un numero solo non può esprimere: la ciclicita del tempo.

In [22]:
# Derivo Ora da Time, come nell'EDA
ora = (df_pulito['Time'] // 3600) % 24

# Trasformo in angolo (giro completo diviso 24) e ricavo le due coordinate
angolo = 2 * np.pi * ora / 24
ora_sin = np.sin(angolo)
ora_cos = np.cos(angolo)

# --- Verifica: 23 e 0 devono risultare VICINE, 0 e 12 LONTANE ---
# Prendo un esempio per alcune ore chiave e guardo le coordinate (sin, cos)
for h in [0, 1, 12, 23]:
    a = 2 * np.pi * h / 24
    print(f"Ora {h:>2}:  sin={np.sin(a):+.3f}  cos={np.cos(a):+.3f}")

Ora  0:  sin=+0.000  cos=+1.000
Ora  1:  sin=+0.259  cos=+0.966
Ora 12:  sin=+0.000  cos=-1.000
Ora 23:  sin=-0.259  cos=+0.966


### Verifica della trasformazione ciclica

Controllo le coordinate (sin, cos) di alcune ore chiave, prima di usarle nel modello:

- Ora 0:  sin +0,000, cos +1,000
- Ora 1:  sin +0,259, cos +0,966
- Ora 12: sin +0,000, cos -1,000
- Ora 23: sin -0,259, cos +0,966

La trasformazione funziona come previsto: **0 e 23** hanno coordinate quasi identiche (adiacenti sul cerchio), mentre **0 e 12** hanno coseno opposto (+1 contro -1: agli antipodi, 12 ore di distanza). Le ore **1 e 23**, equidistanti da mezzanotte da lati opposti, hanno stesso coseno e seno opposto: sono speculari.

Questo mostra anche perche servono **due** coordinate e non una: ore diverse possono condividere lo stesso seno (o lo stesso coseno), e solo la coppia (sin, cos) identifica univocamente ogni ora sul cerchio.

In [23]:
# Parto da df_pulito, aggiungo Ora_sin e Ora_cos derivate da Time
df_ora = df_pulito.copy()
ora = (df_ora['Time'] // 3600) % 24
angolo = 2 * np.pi * ora / 24
df_ora['Ora_sin'] = np.sin(angolo)
df_ora['Ora_cos'] = np.cos(angolo)

# X con le due nuove colonne; tolgo Class, Time (e Ora se presente)
colonne_fuori = [c for c in ['Class', 'Time', 'Ora'] if c in df_ora.columns]
X_ora = df_ora.drop(columns=colonne_fuori)
y_ora = df_ora['Class']

# Stesso split di sempre (stesso random_state -> stesse righe)
X_train_ora, X_test_ora, y_train_ora, y_test_ora = train_test_split(
    X_ora, y_ora, test_size=0.25, stratify=y_ora, random_state=42
)

# Verifica: le due colonne ci sono davvero? quante feature ora?
print("Ci sono Ora_sin e Ora_cos?", 
      'Ora_sin' in X_ora.columns and 'Ora_cos' in X_ora.columns)
print("Numero di feature:", X_ora.shape[1], "(erano 29, ora dovrebbero essere 31)")

Ci sono Ora_sin e Ora_cos? True
Numero di feature: 31 (erano 29, ora dovrebbero essere 31)


In [24]:
# Pipeline identica alla A, addestrata sul dataset CON le colonne orarie
pipeline_ora = Pipeline(steps=[
    ('preprocessore', ColumnTransformer(
        transformers=[('scala_amount', RobustScaler(), ['Amount'])],
        remainder='passthrough'   # V + Ora_sin + Ora_cos passano intatte
    )),
    ('modello', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

pipeline_ora.fit(X_train_ora, y_train_ora)

# Confronto: baseline (senza Ora) vs con Ora
confronto_ora = pd.DataFrame([
    metriche('A: senza Ora',  pipeline_A,    X_test_raw, y_test_raw),
    metriche('D: con Ora',    pipeline_ora,  X_test_ora, y_test_ora),
]).set_index('modello').T

print(confronto_ora.to_string(float_format=lambda x: f'{x:g}'))

modello         A: senza Ora  D: con Ora
frodi_prese              105         105
frodi_sfuggite            13          13
falsi_allarmi           1812        1795
precision              0.055       0.055
recall                  0.89        0.89
ROC_AUC                 0.97       0.971
PR_AUC                 0.678       0.682


### Esito: `Ora` dà un miglioramento piccolo ma coerente

Confronto tra baseline (senza Ora) e modello con `Ora_sin`/`Ora_cos`, a parità di tutto il resto e soglia 0,5:

- falsi allarmi: 1.812 -> 1.795 (17 in meno)
- PR-AUC: 0,678 -> 0,682
- ROC-AUC: 0,970 -> 0,971
- frodi prese, recall, precision: identici

Il miglioramento è piccolo: 17 falsi allarmi in meno su 70.932, e le AUC che si muovono di un soffio. Da solo, un guadagno così esiguo non basterebbe a giustificare l'aggiunta di due colonne.

**Perchè comunque si tiene.** A differenza del log su `Amount` (dove i segnali erano contrastanti - PR-AUC su ma falsi allarmi in aumento), qui i due indicatori concordano: PR-AUC e falsi allarmi migliorano entrambi e, soprattutto, la feature ha un fondamento: l'EDA aveva già dimostrato che l'ora porta segnale (fascia 2-4 con tasso di frode molto più alto). Quindi il miglioramento non è fortuito, ma coerente con ciò che sapevamo dei dati. Su questa base - segnale teoricamente fondato e indicatori concordi - `Ora` si tiene, pur riconoscendo che l'effetto è modesto.

**Nota di metodo.** La decisione non si basa sull'entità del guadagno (troppo piccola per essere decisiva) ma sulla coerenza tra evidenza empirica dell'EDA e comportamento del modello.

## 12. Esperimento: XGBoost al posto della regressione logistica

Fino a qui abbiamo cambiato **i dati** (scaling, log, Ora) tenendo sempre lo stesso modello, la regressione logistica. Questo esperimento cambia invece **il modello**, lasciando i dati come nel baseline.

### Perche XGBoost

La regressione logistica e un modello **lineare**: separa frodi e normali con una combinazione pesata delle feature, in sostanza tracciando un confine 'dritto', ma le frodi possono seguire pattern **non lineari** - combinazioni e interazioni tra feature che una separazione lineare non cattura.

XGBoost e un modello di tipo diverso: si basa su **alberi di decisione** costruiti in sequenza, dove ogni albero corregge gli errori del precedente. Gli alberi decidono per soglie ('questa feature supera un certo valore?') e, combinati, catturano relazioni non lineari e interazioni. E' quindi il primo esperimento che può produrre un salto vero, non un miglioramento al margine.

### Due differenze rispetto alla logistica

**Il bilanciamento cambia nome.** XGBoost non ha `class_weight='balanced'`. Usa `scale_pos_weight`, un numero che indica quante volte pesare la classe rara. La convenzione è il rapporto tra le classi (normali diviso frodi), calcolato dai dati del train (circa 598).

**Lo scaling non servirebbe.** Gli alberi decidono per soglie, non per somme pesate, quindi le scale diverse non li disturbano: `Amount` non scalato non sarebbe un problema. Lo scaler viene comunque mantenuto nella pipeline, per confrontare XGBoost con gli altri esperimenti a parità di preprocessing - cosi l'unica variabile che cambia è il modello.

### Metodo

Si riusa la stessa pipeline del baseline, sostituendo solo il modello (XGBoost al posto della logistica). Preprocessing, split e dati restano identici: ogni differenza nei risultati e attribuibile al cambio di modello. Grazie alla struttura a pipeline, questo cambio è una sola riga - il motivo per cui la pipeline era stata costruita prima degli esperimenti.

In [25]:
# Calcolo scale_pos_weight dai dati: quante volte le normali superano le frodi (nel train)
n_normali = (y_train_raw == 0).sum()
n_frodi   = (y_train_raw == 1).sum()
peso = n_normali / n_frodi
print(f"scale_pos_weight = {peso:.1f}  (normali {n_normali:,} / frodi {n_frodi})")

# Pipeline con XGBoost al posto della logistica — tutto il resto identico alla A
pipeline_xgb = Pipeline(steps=[
    ('preprocessore', ColumnTransformer(
        transformers=[('scala_amount', RobustScaler(), ['Amount'])],
        remainder='passthrough'
    )),
    ('modello', XGBClassifier(
        scale_pos_weight=peso,     # bilanciamento per la classe rara
        n_estimators=200,          # numero di alberi
        max_depth=4,               # profondita di ogni albero (contenuta = evita overfitting)
        learning_rate=0.1,         # quanto "impara" a ogni albero (passo prudente)
        eval_metric='aucpr',       # metrica interna: PR-AUC, adatta a dati sbilanciati
        random_state=42,
    ))
])

pipeline_xgb.fit(X_train_raw, y_train_raw)
print("Pipeline XGBoost addestrata.")

scale_pos_weight = 598.4  (normali 212,439 / frodi 355)
Pipeline XGBoost addestrata.


### Nota sui parametri di XGBoost

I parametri di XGBoost (`max_depth=4`, `n_estimators=200`, `learning_rate=0.1`) sono stati fissati a valori ragionevoli, ma **non ottimizzati**. Controllano la complessità del modello: `max_depth` quanto e profondo ogni albero (più profondo coglie interazioni complesse ma rischia overfitting), `n_estimators` quanti alberi in sequenza, `learning_rate` quanto ciascuno contribuisce.

Trovare la combinazione migliore di questi valori è un'attività a sè, il tuning degli iperparametri (es. GridSearch), che addestra e confronta molti modelli. E' un lavoro corposo che merita un notebook dedicato, quindi, qui non è stato svolto: XGBoost è usato con parametri di default ragionevoli.


In [26]:
confronto_xgb = pd.DataFrame([
    metriche('A: logistica',  pipeline_A,   X_test_raw, y_test_raw),
    metriche('E: XGBoost',    pipeline_xgb, X_test_raw, y_test_raw),
]).set_index('modello').T

print(confronto_xgb.to_string(float_format=lambda x: f'{x:g}'))

modello         A: logistica  E: XGBoost
frodi_prese              105          95
frodi_sfuggite            13          23
falsi_allarmi           1812          36
precision              0.055       0.725
recall                  0.89       0.805
ROC_AUC                 0.97       0.976
PR_AUC                 0.678         0.8


### Esito: XGBoost è il modello migliore, ma va tarata la soglia

Confronto tra la regressione logistica (baseline) e XGBoost, a parità di dati e preprocessing, soglia 0,5:

| metrica | Logistica | XGBoost |
|---|---|---|
| frodi prese | 105 | 95 |
| falsi allarmi | 1.812 | 36 |
| precision | 0,055 | 0,725 |
| recall | 0,890 | 0,805 |
| PR-AUC | 0,678 | 0,800 |

**Cosa e cambiato.** XGBoost ha un carattere completamente diverso dalla logistica. I falsi allarmi crollano da 1.812 a 36 e la precision passa da 0,055 a 0,725: mentre la logistica segnalava frode nel mucchio (solo il 5,5% degli allarmi era vero), quasi 3 allarmi su 4 di XGBoost sono frodi reali. Il prezzo è qualche frode in più sfuggita (95 prese invece di 105, recall da 0,890 a 0,805).

**Il giudice principale conferma XGBoost.** La PR-AUC - che non dipende dalla soglia e riassume il modello a ogni possibile taglio - sale da 0,678 a 0,800. Non è quindi un effetto della soglia scelta: XGBoost, come modello, separa le frodi oggettivamente meglio della logistica. E' il primo esperimento che produce un salto sostanziale, non un miglioramento al margine, perchè per la prima volta cambia il tipo di modello (alberi non lineari) e non i dati.

**Percheè il confronto a soglia 0,5 non basta.** I due modelli a 0,5 stanno su punti operativi diversi: la logistica su 'recall alto, precision bassa', XGBoost su 'precision alta, recall leggermente piu basso'. Confrontarli a soglia fissa e utile per capire quale modello e migliore (e lo è XGBoost), ma non per decidere il punto di utilizzo. Il passo successivo è tarare la soglia di XGBoost: essendo già molto preciso è probabile che, abbassando la soglia, recuperi frodi mantenendo comunque molti meno falsi allarmi della logistica - potenzialmente il meglio dei due mondi.

**Nota su `scale_pos_weight`.** XGBoost gestisce lo sbilanciamento con `scale_pos_weight = 598,4`, cioe il rapporto tra normali e frodi nel **train** (212.439 / 355).

In [27]:
#verifica solo per curiosità e visualizzazione personale
confronto_xgb = pd.DataFrame([
    metriche('A: logistica 0.5',  pipeline_A,   X_test_raw, y_test_raw),
    metriche('A: logistica 0.7',  pipeline_A,   X_test_raw, y_test_raw, soglia=0.7),
    metriche('E: XGBoost 0.5',    pipeline_xgb, X_test_raw, y_test_raw),
]).set_index('modello').T

print(confronto_xgb.to_string(float_format=lambda x: f'{x:g}'))

modello         A: logistica 0.5  A: logistica 0.7  E: XGBoost 0.5
frodi_prese                  105               104              95
frodi_sfuggite                13                14              23
falsi_allarmi               1812               833              36
precision                  0.055             0.111           0.725
recall                      0.89             0.881           0.805
ROC_AUC                     0.97              0.97           0.976
PR_AUC                     0.678             0.678             0.8


In [28]:
soglie = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

tabella_soglie_xgb = pd.DataFrame([
    metriche(f'soglia {s}', pipeline_xgb, X_test_raw, y_test_raw, soglia=s)
    for s in soglie
]).set_index('modello')

print(tabella_soglie_xgb.to_string(float_format=lambda x: f'{x:g}'))

            frodi_prese  frodi_sfuggite  falsi_allarmi  precision  recall  ROC_AUC  PR_AUC
modello                                                                                   
soglia 0.1          100              18            197      0.337   0.847    0.976     0.8
soglia 0.2           98              20            107      0.478   0.831    0.976     0.8
soglia 0.3           97              21             67      0.591   0.822    0.976     0.8
soglia 0.4           96              22             46      0.676   0.814    0.976     0.8
soglia 0.5           95              23             36      0.725   0.805    0.976     0.8
soglia 0.6           93              25             25      0.788   0.788    0.976     0.8
soglia 0.7           92              26             20      0.821    0.78    0.976     0.8
soglia 0.8           92              26             13      0.876    0.78    0.976     0.8
soglia 0.9           92              26             10      0.902    0.78    0.976     0.8

### La soglia operativa di XGBoost: 0,2

A differenza del baseline logistico, XGBoost parte già precisissimo (a soglia 0,5 solo 36 falsi allarmi). Quindi qui la taratura va nella direzione opposta: si **abbassa** la soglia per recuperare frodi, pagando pochissimo in falsi allarmi.

**Perchè non scendere fino a 0,1.** Le ultime frodi sono le più costose da catturare: per prenderle il modello deve abbassare tanto l'asticella da tirar su molte transazioni normali insieme. Il calcolo del 'costo per frode' lo mostra:

- da 0,2 a 0,1: si guadagnano 2 frodi (98 -> 100) ma si pagano 90 falsi allarmi in piu (107 -> 197), cioe **45 falsi allarmi per ogni frode in piu**;
- da 0,3 a 0,2: si guadagna 1 frode ma si pagano 40 falsi allarmi (**40 per frode**).

Il gradino verso 0,1 e il più caro di tutti. Quelle 2 frodi extra raramente valgono 90 verifiche inutili, considerando che i falsi allarmi hanno un costo reale: lavoro di verifica, clienti legittimi disturbati, e la 'fatica da allarme' (se troppi allarmi sono falsi, gli operatori smettono di fidarsi del sistema).

**Perche 0,2.** A soglia 0,2 il modello prende 98 frodi su 118 (l'83%) con soli 107 falsi allarmi e precision 0,478. Rispetto alla logistica al suo punto migliore (soglia 0,7: 104 frodi, 833 falsi allarmi), XGBoost a 0,2 prende quasi le stesse frodi con **un ottavo** dei falsi allarmi. E' il 'meglio dei due mondi': recall vicino a quello della logistica, ma con una frazione minima dei falsi allarmi.

In altre parole: scendere di soglia fa sempre guadagnare qualche frode in più, ma più si scende, più ogni frode aggiuntiva costa in falsi allarmi. A 0,2 il costo e ancora ragionevole (40 falsi allarmi per frode); scendere a 0,1 lo fa salire a 45 per appena 2 frodi. 0,2 e quindi il punto oltre il quale continuare a scendere non conviene più: si pagherebbe troppo in falsi allarmi per troppo poche frodi in più.

In [29]:
# Costo di ogni gradino: quanti falsi allarmi in più per ogni frode in più, scendendo di soglia
t = tabella_soglie_xgb.copy()
t['frodi_guadagnate']    = t['frodi_prese'].diff() * -1
t['falsi_allarmi_extra'] = t['falsi_allarmi'].diff() * -1

# costo per frode solo dove si guadagnano davvero frodi; altrimenti "-"
t['costo_per_frode'] = t.apply(
    lambda r: round(r['falsi_allarmi_extra'] / r['frodi_guadagnate'], 1)
              if r['frodi_guadagnate'] > 0 else '-',
    axis=1
)

print(t[['frodi_prese', 'falsi_allarmi', 'frodi_guadagnate', 'falsi_allarmi_extra', 'costo_per_frode']].to_string())

            frodi_prese  falsi_allarmi  frodi_guadagnate  falsi_allarmi_extra costo_per_frode
modello                                                                                      
soglia 0.1          100            197               NaN                  NaN               -
soglia 0.2           98            107               2.0                 90.0            45.0
soglia 0.3           97             67               1.0                 40.0            40.0
soglia 0.4           96             46               1.0                 21.0            21.0
soglia 0.5           95             36               1.0                 10.0            10.0
soglia 0.6           93             25               2.0                 11.0             5.5
soglia 0.7           92             20               1.0                  5.0             5.0
soglia 0.8           92             13              -0.0                  7.0               -
soglia 0.9           92             10              -0.0    

### Il costo marginale di ogni soglia

Per rendere quantitativa la scelta della soglia, calcolo quanto costa ogni gradino: scendendo di soglia, quanti falsi allarmi in più pago per ogni frode in più che guadagno.

La colonna `costo_per_frode` mostra un andamento chiaro: le prime frodi (soglie alte) costano pochissimo - intorno a 0,6-0,7 bastano 5 falsi allarmi per frode - mentre le ultime (soglie basse) costano molto, fino a 45 per frode nel salto verso 0,1. E' la curva dei rendimenti decrescenti resa numero: più si abbassa la soglia, più ogni frode aggiuntiva costa in falsi allarmi.

Perchè allora scegliere 0,2 e non una soglia più alta col costo per frode piu basso (0,7 costa appena 5)? 
Perchè la scelta bilancia due obiettivi opposti: 
1. prendere piu frodi possibile (recall alto) 
2. non pagarle troppo care (costo per frode basso). 
Scendere di soglia fa salire il recall, quindi, conviene scendere, ma solo finchè il costo resta ragionevole.

A 0,2 il recall è alto (98 frodi su 118) e il costo ancora accettabile. Il gradino successivo, verso 0,1, costerebbe 45 falsi allarmi per sole 2 frodi in più: il piu caro della tabella. Quindi 0,2 e il punto piu basso conveniente - si scende per massimizzare le frodi prese, e ci si ferma appena prima che il costo diventi proibitivo. Non è la soglia col costo piu basso in assoluto (lo sarebbe 0,7), ma il miglior compromesso tra frodi catturate e falsi allarmi pagati.

Nota: dove le frodi prese non cambiano (soglie 0,8-0,9), scendere di soglia non guadagna nessuna frode, quindi il costo per frode non si applica ed e indicato con "-".

### Il modello finale: XGBoost con soglia 0,2

Fisso la configurazione scelta - XGBoost, soglia 0,2 - e ne fotografo la matrice di confusione e il report, come modello definitivo di questo notebook.

In [30]:
# Ricalcolo le probabilita di XGBoost sul test (serve per applicare la soglia)
y_prob_xgb = pipeline_xgb.predict_proba(X_test_raw)[:, 1]

In [31]:
SOGLIA_XGB = 0.2

y_pred_finale = (y_prob_xgb >= SOGLIA_XGB).astype(int)

print(f"MODELLO FINALE — XGBoost, soglia {SOGLIA_XGB}\n")
cm = confusion_matrix(y_test_raw, y_pred_finale)
print(f"                    Prev. Normale    Prev. Frode")
print(f"Reale Normale         {cm[0,0]:>8,}      {cm[0,1]:>7,}")
print(f"Reale Frode           {cm[1,0]:>8,}      {cm[1,1]:>7,}")
print()
print(classification_report(y_test_raw, y_pred_finale, target_names=['Normale', 'Frode'], digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test_raw, y_prob_xgb):.3f}  (invariata: non dipende dalla soglia)")
print(f"PR-AUC:  {average_precision_score(y_test_raw, y_prob_xgb):.3f}  (invariata: non dipende dalla soglia)")

MODELLO FINALE — XGBoost, soglia 0.2

                    Prev. Normale    Prev. Frode
Reale Normale           70,707          107
Reale Frode                 20           98

              precision    recall  f1-score   support

     Normale      1.000     0.998     0.999     70814
       Frode      0.478     0.831     0.607       118

    accuracy                          0.998     70932
   macro avg      0.739     0.914     0.803     70932
weighted avg      0.999     0.998     0.998     70932

ROC-AUC: 0.976  (invariata: non dipende dalla soglia)
PR-AUC:  0.800  (invariata: non dipende dalla soglia)


## 13. Riepilogo: il percorso completo

Questa tabella mette a confronto tutti i modelli costruiti nel notebook, ciascuno al suo punto operativo, per rendere visibile il percorso dal baseline al modello finale.

In [32]:
riepilogo = pd.DataFrame([
    metriche('Logistica 0.5',        pipeline_A,   X_test_raw, y_test_raw),
    metriche('Logistica 0.7',        pipeline_A,   X_test_raw, y_test_raw, soglia=0.7),
    metriche('Logistica+Ora 0.5',    pipeline_ora, X_test_ora, y_test_ora),
    metriche('XGBoost 0.5',          pipeline_xgb, X_test_raw, y_test_raw),
    metriche('XGBoost 0.2 (finale)', pipeline_xgb, X_test_raw, y_test_raw, soglia=0.2),
]).set_index('modello')

riepilogo.style.background_gradient(cmap='Greens', subset=['PR_AUC', 'precision']).format({'precision': '{:.3f}', 'recall': '{:.3f}', 'ROC_AUC': '{:.3f}', 'PR_AUC': '{:.3f}'})

#riepilogo   

,frodi_prese,frodi_sfuggite,falsi_allarmi,precision,recall,ROC_AUC,PR_AUC
modello,,,,,,,
Logistica 0.5,105,13,1812,0.055,0.890,0.970,0.678
Logistica 0.7,104,14,833,0.111,0.881,0.970,0.678
Logistica+Ora 0.5,105,13,1795,0.055,0.890,0.971,0.682
XGBoost 0.5,95,23,36,0.725,0.805,0.976,0.800
XGBoost 0.2 (finale),98,20,107,0.478,0.831,0.976,0.800


## 14. Conclusione

Il notebook è partito da un baseline lineare ed è arrivato a un modello nettamente migliore, misurando ogni scelta invece di assumerla.

**Il percorso in sintesi.** La regressione logistica (baseline) prendeva l'89% delle frodi ma con precision del 5,5%: quasi tutti i suoi allarmi erano falsi. La taratura della soglia e l'aggiunta di `Ora` hanno prodotto miglioramenti reali ma marginali. Il salto vero e arrivato cambiando modello: XGBoost, non lineare, ha portato la PR-AUC da 0,678 a 0,800 e ridotto i falsi allarmi da migliaia a poche decine.

**Il modello finale** - XGBoost con soglia 0,2 - prende 98 frodi su 118 (recall 83%) con soli 107 falsi allarmi (precision 48%). L'F1 sulla classe frode passa da 0,10 del baseline a 0,61: sei volte migliore. Rispetto alla logistica al suo punto migliore (104 frodi, 833 falsi allarmi), XGBoost prende quasi le stesse frodi con un ottavo dei falsi allarmi.

**Cosa è stato imparato lungo il percorso.** Diversi esperimenti hanno dato esito negativo, ed è stato utile quanto quelli positivi: scalare tutte le feature invece del solo `Amount` non cambia nulla (le V escono già scalate dalla PCA); la trasformazione logaritmica di `Amount` non migliora il modello (Amount è una feature marginale, come confermano anche i coefficienti della logistica). Sapere cosa non serve e perchè è parte del risultato.

**Metodo.** Ogni scelta - soglia, scaling, feature, modello - è stata verificata sui dati e non assunta: verifiche di coerenza dopo ogni passaggio, confronti a parità di condizioni, decisioni motivate. Il valore del modello (PR-AUC) è stato distinto dal suo punto operativo (la soglia), scelto in base al costo relativo tra frode persa e falso allarme.

**Prossimi passi:**

Notebook successivi
1. **Notebook 03 - Impatto della qualità del dato**: riaddestrare il modello sui dati con i duplicati e confrontarlo con quello sui dati puliti, a parità di tutto il resto, per quantificare quanto la pulizia cambia i risultati. Usa la pipeline costruita qui.
2. **Notebook 04 - Interpretabilità (SHAP)**: spiegare le predizioni di XGBoost con SHAP, che indica quanto ogni feature contribuisce a ogni singola decisione. Rende spiegabile un modello altrimenti a scatola nera - importante in ambito antifrode, dove le decisioni vanno giustificate.
3. **Notebook 05 - Tuning degli iperparametri**: ricerca sistematica (es. GridSearch) dei valori migliori di max_depth, n_estimators, learning_rate, per spremere ulteriormente XGBoost, che qui è stato usato con parametri ragionevoli ma non ottimizzati.

Esperimenti aggiuntivi

4. **SMOTE**: testare l'oversampling sintetico della classe rara come alternativa a `scale_pos_weight`, e confrontarne l'effetto.
5. **Altri modelli**: Random Forest come ulteriore termine di paragone contro logistica e XGBoost.
6. **`Ora` con soglia ottimizzata**: `Ora` è stata valutata solo a soglia 0,5; una volta scelto il modello, si potrebbe rivalutare il suo apporto tarando anche la soglia.
7. **Feature engineering su `Amount`**: pur avendo un ruolo marginale, si potrebbero testare altre trasformazioni o interazioni (es. Amount combinato con Ora) per verificarne l'utilità.